# SSGF Autopoietic Closure Loop

The Strange-loop Self-Generating Field (SSGF) implements a cybernetic
feedback cycle where geometry produces dynamics which produce cost
which produces gradient which updates geometry:

```
z (latent) -> W (coupling) -> Kuramoto cost -> grad dU/dz -> z update
     ^                                                          |
     +----------------------------------------------------------+
```

**GeometryCarrier** holds a latent vector `z` and decodes it into a
coupling matrix `W(z) = softplus(A @ z)` via a fixed random projection.

**CyberneticClosure** runs the loop: observe phases, compute SSGF costs
(C1-C4), take a gradient step on z to minimize U_total, decode new W.

Cost terms:
- **C1** (sync deficit): `1 - R` -- penalizes desynchronization
- **C2** (spectral gap): `-lambda_2(L)` -- maximizes algebraic connectivity
- **C3** (sparsity): `||W||_1 / N^2` -- prevents dense coupling
- **C4** (symmetry): `||W - W^T||_F / N` -- penalizes asymmetry

In [ ]:
import numpy as np

from scpn_phase_orchestrator.ssgf.carrier import GeometryCarrier
from scpn_phase_orchestrator.ssgf.closure import CyberneticClosure
from scpn_phase_orchestrator.ssgf.costs import compute_ssgf_costs
from scpn_phase_orchestrator.upde.order_params import compute_order_parameter

In [ ]:
# Create GeometryCarrier for 6 oscillators
N_OSC = 6
Z_DIM = 10
LR = 0.02

carrier = GeometryCarrier(n_oscillators=N_OSC, z_dim=Z_DIM, lr=LR, seed=42)

print(f"Latent z dimension: {carrier.z_dim}")
print(f"Initial z (first 5): {carrier.z[:5].round(4)}")

W_init = carrier.decode()
print(f"\nInitial W shape: {W_init.shape}")
print("Initial W:")
print(np.round(W_init, 4))

In [ ]:
# Run CyberneticClosure for 20 outer steps
# Use partially synchronized phases as the target state
rng = np.random.default_rng(7)
phases = rng.uniform(0, 2 * np.pi, N_OSC)
# Bring half the oscillators close together to give the optimizer a signal
phases[:3] = phases[0] + rng.normal(0, 0.1, 3)

R_before, _ = compute_order_parameter(phases)
print(f"Phase R before optimization: {R_before:.4f}")

closure = CyberneticClosure(
    carrier=carrier,
    cost_weights=(1.0, 0.5, 0.1, 0.1),
)

N_OUTER = 20
W_final, history = closure.run(phases, n_outer_steps=N_OUTER)

# Print convergence
print(f"\nClosure ran {len(history)} steps")
for _i, cs in enumerate(history):
    tag = "OK" if cs.converging else "!!"
    before = f"{cs.cost_before:.4f}"
    after = f"{cs.cost_after:.4f}"
    print(f"  step {cs.ssgf_state_step:2d}: cost {before} -> {after}  [{tag}]")

try:
    import matplotlib.pyplot as plt

    costs_before = [cs.cost_before for cs in history]
    costs_after = [cs.cost_after for cs in history]
    steps = range(1, len(history) + 1)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(steps, costs_before, "o-", label="cost_before", markersize=4)
    ax.plot(steps, costs_after, "s-", label="cost_after", markersize=4)
    ax.set_xlabel("Outer step")
    ax.set_ylabel("U_total")
    ax.set_title("SSGF Closure: Cost Descent")
    ax.legend()
    fig.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed, skipping plot")

In [ ]:
# Compare W matrix before and after optimization
print("W BEFORE optimization:")
print(np.round(W_init, 4))
print("\nW AFTER optimization:")
print(np.round(W_final, 4))

# Structural changes
delta_W = W_final - W_init
print(f"\nMax |delta W|: {np.max(np.abs(delta_W)):.4f}")
print(f"Frobenius norm of delta W: {np.linalg.norm(delta_W, 'fro'):.4f}")

try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    im0 = axes[0].imshow(W_init, cmap="viridis", vmin=0)
    axes[0].set_title("W (initial)")
    plt.colorbar(im0, ax=axes[0], fraction=0.046)

    im1 = axes[1].imshow(W_final, cmap="viridis", vmin=0)
    axes[1].set_title("W (after 20 steps)")
    plt.colorbar(im1, ax=axes[1], fraction=0.046)

    vmax = max(abs(delta_W.min()), abs(delta_W.max()))
    im2 = axes[2].imshow(delta_W, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    axes[2].set_title("delta W")
    plt.colorbar(im2, ax=axes[2], fraction=0.046)

    fig.suptitle("Coupling Geometry Before/After SSGF Closure")
    fig.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed, skipping plot")

In [ ]:
# SSGF cost breakdown (C1-C4) before and after
costs_init = compute_ssgf_costs(W_init, phases, weights=(1.0, 0.5, 0.1, 0.1))
costs_final = compute_ssgf_costs(W_final, phases, weights=(1.0, 0.5, 0.1, 0.1))

print("Cost breakdown:")
print(f"{'Term':<20} {'Before':>10} {'After':>10} {'Delta':>10}")
print("-" * 52)
for name in ["c1_sync", "c2_spectral_gap", "c3_sparsity", "c4_symmetry", "u_total"]:
    before = getattr(costs_init, name)
    after = getattr(costs_final, name)
    delta = after - before
    print(f"{name:<20} {before:10.4f} {after:10.4f} {delta:+10.4f}")